In [31]:
import pandas as pd
import numpy as np
import os

folder_path = r'C:\Users\abhishek\OneDrive\Documents\Bangalore-B2B-Intelligence-Engine\Data\Raw Data'
print("Starting the Data Refinery Pipeline...\n")

try:
    all_files = os.listdir(folder_path)
    excel_files = [f for f in all_files if f.endswith('.xlsx') and 'stores' in f.lower()]
    
    if len(excel_files) == 0:
        print("Error: Could not find the stores.xlsx file in your folder.")
    else:
        file_name = excel_files[0]
        file_path = os.path.join(folder_path, file_name)
        print(f"Success! Loaded file: {file_name}")
        
        df = pd.read_excel(file_path, engine='openpyxl')
        
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
        print(f"Original Data Shape: {df.shape}")

        def clean_and_explode(dataframe):
            df_clean = dataframe.copy()
            
            def safe_split(val):
                if pd.isna(val): 
                    return ['']
                return str(val).split('\n')
            
            for col in df_clean.columns:
                df_clean[col] = df_clean[col].apply(safe_split)
            
            def match_lengths(row):
                max_len = max(len(lst) for lst in row)
                for col in row.index:
                    if len(row[col]) < max_len:
                        row[col] = row[col] + [''] * (max_len - len(row[col]))
                return row
            
            df_clean = df_clean.apply(match_lengths, axis=1)
            
            df_clean = df_clean.explode(column=list(df_clean.columns))
            df_clean = df_clean.reset_index(drop=True)
            
            df_clean = df_clean.replace('nan', '')
            
            return df_clean

        df_exploded = clean_and_explode(df)
        print(f"New Data Shape after un-jamming rows: {df_exploded.shape}")

        df_exploded['PHONENUMBER'] = df_exploded['PHONENUMBER'].astype(str).str.replace(r'\D', '', regex=True)
        df_exploded['PHONE_LENGTH'] = df_exploded['PHONENUMBER'].str.len()

        valid_phones = df_exploded[df_exploded['PHONE_LENGTH'] == 10]
        invalid_phones = df_exploded[df_exploded['PHONE_LENGTH'] != 10]

        print("\n" + "="*40)
        print("DATABASE QUALITY SCORECARD")
        print("="*40)
        print(f"Total Stores Extracted: {len(df_exploded)}")
        print(f"Perfect 10-digit Phone Numbers: {len(valid_phones)}")
        print(f"Broken/Missing Phone Numbers: {len(invalid_phones)}")
        print("="*40 + "\n")

        display(df_exploded.head(15))
        
except Exception as e:
    print(f"A system error occurred: {e}")

Starting the Data Refinery Pipeline...

Success! Loaded file: stores.xlsx
Original Data Shape: (735, 5)
New Data Shape after un-jamming rows: (950, 5)

DATABASE QUALITY SCORECARD
Total Stores Extracted: 950
Perfect 10-digit Phone Numbers: 725
Broken/Missing Phone Numbers: 225



,SLNO,NAME,PHONENUMBER,LOCATION,APPLICATION,PHONE_LENGTH
0,1,Mangalore,9742074219,1st stage MSR Nagar Mathikere,General Store,10
1,2,Suresh,8023607471,10th cross MSR Nagar Mathikere,General Store,10
2,3,Smitha,8040913427,11th cross MSR Nagar Mathikere,General Store,10
3,4,Pigoen,8049706614,7th cross MSR Nagar Mathikere,General Store,10
4,5,Anugraha,8023006615,10th cross MSR Nagar Mathikere,General Store,10
5,6,Kraze,9845131423,1st cross MSR Nagar Mathikere,General Store,10
6,7,More,8108138000,3rd cross MSR Nagar Mathikere,General Store,10
7,8,Subramanya,8495936677,1st cross MSR Nagar Mathikere,General Store,10
8,9,Bhawaralal,9844223316,2nd cross MSR Nagar Mathikere,General Store,10
9,10,Mamtha,7339997418,HMT layout Mathikere,General Store,10


In [33]:

df_exploded.loc[df_exploded['PHONE_LENGTH'] != 10, 'PHONENUMBER'] = 'Unknown'
df_final = df_exploded.drop(columns=['PHONE_LENGTH'])
output_folder = r'C:\Users\abhishek\OneDrive\Documents\Bangalore-B2B-Intelligence-Engine\Data\Processed'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

output_file = os.path.join(output_folder, 'clean_stores.csv')
df_final.to_csv(output_file, index=False)

print(f"Your clean data has been permanently saved to:")
print(output_file)

Your clean data has been permanently saved to:
C:\Users\abhishek\OneDrive\Documents\Bangalore-B2B-Intelligence-Engine\Data\Processed\clean_stores.csv
